<a href="https://colab.research.google.com/github/dranphphmithe-ux/Book-Rental-System-Project/blob/Nam-Wan/%E0%B8%AA%E0%B8%B3%E0%B9%80%E0%B8%99%E0%B8%B2%E0%B8%82%E0%B8%AD%E0%B8%87_book_rental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:

class Receipt:   #ของน้ำหวาน
    """คลาสคำนวณเงินและออกใบเสร็จ (ปรับโปรโมชั่นเป็น: แถมวันยืมเพิ่มฟรี)"""

    import pandas as pd
import random
from datetime import datetime

# 1. โหลดข้อมูลจาก CSV
url = 'https://raw.githubusercontent.com/dranphphmithe-ux/Book-Rental-System-Project/refs/heads/main/books_cleaned.csv'
df_books = pd.read_csv(url)
df_books.columns = df_books.columns.str.strip()

# ตรวจสอบและแปลงคอลัมน์สต็อกให้เป็นตัวเลข
stock_col = 'stock_qty' if 'stock_qty' in df_books.columns else 'stock'
df_books[stock_col] = pd.to_numeric(df_books[stock_col], errors='coerce').fillna(10).astype(int)

# รายชื่อลูกค้า
FIRST_NAMES = ["กิตติพงษ์", "ณิชา", "ธนกฤต", "ปรียา", "พงศกร", "ภัทรวดี", "วรวุฒิ", "ศิริพร", "อัครพล", "อนันดา"]
LAST_NAMES = ["ใจดี", "เจริญสุข", "สมบูรณ์", "วงษ์สุวรรณ", "รัตนไพศาล", "พงษ์พาณิชย์", "ชินวัตร", "ทองแท้", "สุวรรณรัตน์", "มั่นคง"]

# ==========================================
# ฟังก์ชัน 1: ยืมหนังสือ (ตัดสต็อก + ออกใบเสร็จ)
# ==========================================
def rent_books_and_print_receipt(customer_name, customer_id, initial_points, selected_indices, days_rented, days_late):
    global df_books

    order_id = f"REC-{datetime.now().strftime('%Y%m%d')}-{random.randint(1000, 9999)}"
    date_issued = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    items = []
    # ตัดสต็อกในคลังข้อมูล
    for idx in selected_indices:
        title = df_books.loc[idx, 'title'] if 'title' in df_books.columns else df_books.loc[idx, 'name']
        price = float(df_books.loc[idx, 'price'])

        current_stock = df_books.loc[idx, stock_col]
        new_stock = max(0, current_stock - 1)
        df_books.loc[idx, stock_col] = new_stock

        items.append({
            'title': title,
            'price': price,
            'remaining_stock': new_stock
        })

    total_books = len(items)

    # คำนวณค่ายืม (3 วัน 20 บาท, เศษวันละ 7 บาท)
    sets_of_3 = days_rented // 3
    remaining_days = days_rented % 3
    rental_fee_per_book = (sets_of_3 * 20) + (remaining_days * 7)
    total_rental_before_discount = total_books * rental_fee_per_book

    # แต้มและส่วนลด (ยืม 1 เล่ม = 1 แต้ม, คืนตรงเวลา +1 แต้ม)
    base_points = total_books
    on_time_bonus = 1 if days_late == 0 else 0
    earned_points = base_points + on_time_bonus
    total_points_accumulated = initial_points + earned_points

    # ครบ 10 แต้ม ได้อ่านฟรี 1 วัน (ลด 7 บาท/เล่ม)
    free_books_count = min(total_points_accumulated // 10, total_books)
    total_discount = free_books_count * 7.0

    # ค่าปรับและสรุปยอด
    total_rental_fee = max(0.0, total_rental_before_discount - total_discount)
    total_fine = total_books * days_late * 10
    grand_total = total_rental_fee + total_fine

    points_used = free_books_count * 10
    final_points = total_points_accumulated - points_used

    # พิมพ์ใบเสร็จยืม
    print("=" * 60)
    print(f"{'ใบเสร็จรับเงิน (ยืมหนังสือ) / Rental Receipt':^60}")
    print("=" * 60)
    print(f"เลขที่ใบเสร็จ: {order_id}")
    print(f"วันที่ทำรายการ: {date_issued}")
    print(f"ชื่อลูกค้า: {customer_name} (ID: {customer_id})")
    print(f"จำนวนวันที่ยืม: {days_rented} วัน | คืนช้า: {days_late} วัน")
    print("-" * 60)

    print(f"รายการหนังสือที่ยืม ({total_books} เล่ม):")
    for i, item in enumerate(items, 1):
        print(f"  [{i:02d}] {item['title']} (ราคาปก {item['price']:.0f} บาท) | สต็อกหลังหัก: {item['remaining_stock']} เล่ม")

    print("-" * 60)
    print(f"อัตราค่ายืมปกติต่อเล่ม ({days_rented} วัน): {rental_fee_per_book:.2f} บาท")

    if free_books_count > 0:
        print(f"ส่วนลดสะสมแต้ม (ครบ 10 แต้ม): อ่านฟรี 1 วัน จำนวน {free_books_count} เล่ม (-{total_discount:.2f} บาท)")

    print(f"รวมค่ายืมหนังสือหลังหักส่วนลด: {total_rental_fee:.2f} บาท")

    if total_fine > 0:
        print(f"ค่าปรับคืนช้า ({days_late} วัน x {total_books} เล่ม x 10B): {total_fine:.2f} บาท")

    print("-" * 60)
    print(f"ยอดชำระสุทธิ (Grand Total): {grand_total:.2f} บาท")
    print("-" * 60)

    print(f"แต้มที่ได้รับจากจำนวนหนังสือ (1 เล่ม = 1 แต้ม): +{base_points} แต้ม")
    if days_late == 0:
        print(f"โบนัสคืนตรงเวลา: +{on_time_bonus} แต้ม")
    else:
        print("คืนช้ากว่ากำหนด: ไม่ได้รับโบนัสคืนตรงเวลา (+0 แต้ม)")

    if points_used > 0:
        print(f"ใช้แต้มแลกอ่านฟรี: -{points_used} แต้ม")

    print(f"แต้มสะสมคงเหลือปัจจุบัน: {final_points} แต้ม")
    print("=" * 60 + "\n")